In [ ]:
from pathlib import Path

from env_optimiser import EnvOptimiser
from feature_extractor import MinigridFeaturesExtractor
from procedural_level import ProceduralLevel
from minigrid.wrappers import ImgObsWrapper,OneHotPartialObsWrapper
from custom_callback import CustomCallback
from sb3_contrib import RecurrentPPO
import os

In [ ]:
n_envs = os.cpu_count() or 4
n_timesteps = 128_000
freq = 2000
eval_freq = max(freq // n_envs, 1)  # accounting for multiple environments

save_dir = Path("models/")
save_dir.mkdir(parents=True, exist_ok=True)

env = ProceduralLevel(difficulty=1000, max_steps=120)
optimiser = EnvOptimiser(env=env, n_envs=n_envs, wrapper_cls=[OneHotPartialObsWrapper,ImgObsWrapper], save_dir=save_dir)
vec_env_train = optimiser.build_vec_env()
policy_kwargs = {
    "features_extractor_class": MinigridFeaturesExtractor,
    "features_extractor_kwargs": {"features_dim": 128},
    "normalize_images": False,
}
model_dir = optimiser.file_path.parent  # makes use of the same folder
callback = CustomCallback(check_freq=eval_freq, save_dir=model_dir, verbose=1)
model = RecurrentPPO(policy="CnnLstmPolicy", env=vec_env_train, policy_kwargs=policy_kwargs)
model.learn(total_timesteps=n_timesteps, callback=callback, progress_bar=True)